# Phase 4a — training data v1 (instructions, verification, splits)

Thin Colab runner. The task candidates (`data/sft/candidates.jsonl`) were built locally from the Polars 2.0 docs (docstring examples + user-guide snippets), validated in the sandbox, and carry their source doc chunk(s). Here an **open** instruct model (`Qwen/Qwen3.5-9B`, never Claude) writes two requests per candidate; then quality filters and leakage control run, a **round-trip check** keeps only requests the 9B model can turn back into the reference output (with the source docs as context), the page-grouped 90/10 split is made, and a robustness check re-ranks the retrieval variants on the dev tasks (report only; the frozen B2 config is not changed).

**Runtime:** GPU, target **A100** (steps 1 and 3 run the 9B model; an L4 works with smaller batches, just slower). Both 9B steps checkpoint to Google Drive after every batch and resume from it: step 1 only writes requests for candidates not yet in its checkpoint, step 3 only solves requests not yet in its checkpoint.

**Crash-safe:** if the session dies, reconnect and run all cells again.

In [ ]:
# Confirm the GPU (A100 40 GB targeted; L4 24 GB works with smaller batches).
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
# Less fragmentation for the large writer model; inherited by every `!python` below.
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

In [ ]:
# Clone the repo (idempotent) and enter it.
import os
REPO = 'https://github.com/yusuftanriverdi24/Polars-Rag---Finetune.git'
if not os.path.isdir('polars-rag-vs-ft'):
    !git clone -q $REPO polars-rag-vs-ft
%cd polars-rag-vs-ft
!git pull -q

In [ ]:
# Install pinned dependencies: `colab` = GPU stack, `rag` = sentence-transformers +
# LanceDB (exact pins). torch stays Colab's preinstalled CUDA build.
!pip install -q -e ".[colab,rag]"
# Fast linear-attention kernels for Qwen3.5 (causal-conv1d + flash-linear-attention), PREBUILT WHEELS ONLY:
# installed only if a wheel exists for this torch/CUDA/Python, never compiled; otherwise the model runs on
# the slower torch fallback. The path taken is printed and logged.
!python scripts/install_fast_kernels.py --log results/datagen/fast_kernels.json

In [ ]:
# Google Drive: outputs are copied here so they survive the Colab session.
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/polars-rag-vs-ft'
!mkdir -p $DRIVE_DIR

In [ ]:
# 1. Two requests per candidate (concise + descriptive) with Qwen3.5-9B, bf16, greedy.
#    REUSES the checkpoint on Drive: only candidates missing from it are written (after the part-1 rerun,
#    the new/changed candidates); if none are missing the writer is not even loaded. Starting batch size
#    from free GPU memory, halved on OOM; every finished batch appended to the checkpoint.
CKPT_DIR = DRIVE_DIR + '/datagen'
!mkdir -p "$CKPT_DIR"
# The meta of the run that wrote the checkpoint (kept on Drive; not in the repo).
!cp -n "$CKPT_DIR/instructions_meta.json" results/datagen/ 2>/dev/null || true
!python -m polars_llm.datagen.instructions --checkpoint "$CKPT_DIR/instructions_checkpoint.jsonl"
!cp data/sft/instructions_raw.jsonl results/datagen/instructions_meta.json "$CKPT_DIR/"
!python -c "import json; m=json.load(open('results/datagen/instructions_meta.json')); print(m['gpu'], 'batch sizes', m['batch_sizes_used'], 'parsed', m['n_parsed'], '/', m['n_candidates'])"

In [ ]:
# 2. Filters (method names / code / length / missing columns), then leakage vs the benchmark: a request is
#    dropped only if its granite cosine to a benchmark instruction is >= the calibrated threshold (max
#    nearest-neighbour cosine between distinct benchmark tasks) AND its solution has that task's non-core
#    API set -> data/sft/filtered.jsonl, results/datagen/sft_stats.json.
!python -m polars_llm.datagen.build_sft --stage filter --device cuda
!python -c "import json; s=json.load(open('results/datagen/sft_stats.json')); l=s['instruction_leakage']; print('filters', s['filter_problems']); print('threshold', l['threshold'], 'leakage dropped', l['dropped']); print('after filters + leakage', s['after_filters_and_leakage']['n'])"

In [ ]:
# 3. Round-trip verification (data quality): Qwen3.5-9B (not the 4B student) gets each request + setup +
#    signature + the example's source doc chunk(s) WITHOUT the example itself (no Examples section / own code
#    block; refuses to run if any answer is in its context). Greedy. Its code runs in the sandbox; the request
#    is kept only if the output equals the reference under the candidate's own check flags (as the benchmark).
#    Pinned canaries run alongside. Checkpoint on Drive, adaptive batching. -> data/sft/verified.jsonl,
#    results/datagen/roundtrip_results.jsonl, roundtrip_stats.json. Decision rules: PLAN.md (pre-registered).
!python -m polars_llm.datagen.roundtrip --checkpoint "$CKPT_DIR/roundtrip_checkpoint.jsonl" --workers 8
!cp results/datagen/roundtrip_stats.json results/datagen/roundtrip_results.jsonl "$CKPT_DIR/"
!python -c "import json; s=json.load(open('results/datagen/roundtrip_stats.json')); print('answer in context', s['answer_in_context']['with_answer_in_context']); print('kept', s['kept'], '/', s['n'], s['retention'], s['reasons']); [print(k, {g: v['retention'] for g, v in s[k].items()}) for k in ('by_source', 'by_style', 'by_category', 'by_multi_step', 'by_check_flags')]; print('canaries', s['canaries']['passed'], '/', s['canaries']['n'], [(c['id'], c['expected'], c['actual']) for c in s['canaries']['results']]); print('decision rules', s['decision_rules'])"

In [ ]:
# 4. Page-grouped 90/10 split of the verified examples -> data/sft/train.jsonl, dev.jsonl (+ stats).
!python -m polars_llm.datagen.build_sft --stage split
!python -c "import json; s=json.load(open('results/datagen/sft_stats.json')); print('train', s['train']['n'], 'dev', s['dev']['n']); print('train by category', s['train']['by_category'])"

In [ ]:
# 5. Robustness check of the RAG choice (selection bias): rebuild the indexes, let the base model draft the
#    dev tasks zero-shot, and compare API recall of naive / a / b / c on them. REPORT ONLY.
!python -m polars_llm.rag.ingest
!python -m polars_llm.rag.chunk --stats /tmp/chunk_stats_v1.json
!python -m polars_llm.rag.corpus --stats /tmp/corpus_v2_stats.json
!python -m polars_llm.rag.index --device cuda --batch-size 32
!python -m polars_llm.rag.index --chunks data/rag/chunks_v2.jsonl --index-dir data/rag/index_v2_granite --embedder granite --device cuda --batch-size 32
!python -m polars_llm.rag.index --chunks data/rag/chunks_v2.jsonl --index-dir data/rag/index_v2_qwen3 --embedder qwen3 --device cuda --batch-size 16

In [ ]:
!python scripts/generate.py --arm dev_drafts --model Qwen/Qwen3.5-4B --tasks data/sft/dev.jsonl --out-dir results/datagen --batch-size 8
!python -m polars_llm.rag.eval_retrieval_v2 --device cuda --tasks data/sft/dev.jsonl --drafts results/datagen/dev_drafts/generations.jsonl --out results/datagen/retrieval_robustness_dev.json --frozen-config results/rag/retrieval_eval_v2.json

In [ ]:
# 6. Zip for download (and keep a copy on Drive for notebook 06).
!zip -q -r results_datagen.zip data/sft/train.jsonl data/sft/dev.jsonl data/sft/filtered.jsonl data/sft/verified.jsonl data/sft/instructions_raw.jsonl results/datagen
!cp results_datagen.zip $DRIVE_DIR/
from google.colab import files
files.download('results_datagen.zip')